<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/08-window-functions/01-ranking.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 8.1 — Ranking: row_number, rank, dense_rank, ntile

One ordered window, four functions, one difference: ties. Then the
top-N-per-group pattern and the unique-orderBy discipline.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("8.1-ranking")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = (
    spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
    .withColumn("revenue", F.round(col("quantity") * col("unit_price"), 2))
)

## The family side by side

electronics has enough rows to show every behavior — including the
tie the duplicated order 1021 creates (two rows, same revenue).

In [ ]:
w = Window.partitionBy("category").orderBy(F.desc("revenue"))

ranked = orders.filter(col("category") == "electronics").select(
    "product", "revenue",
    F.row_number().over(w).alias("row_number"),
    F.rank().over(w).alias("rank"),
    F.dense_rank().over(w).alias("dense_rank"),
    F.round(F.percent_rank().over(w), 3).alias("pct_rank"),
    F.ntile(4).over(w).alias("quartile"),
)
ranked.show(20)
# find the 1021 twins: same revenue -> same rank/dense_rank,
# DIFFERENT row_number — and which twin got the smaller one is luck

## Ties change the counts: ==1 under each function

In [ ]:
for fn_name, fn in [("row_number", F.row_number),
                    ("rank", F.rank),
                    ("dense_rank", F.dense_rank)]:
    n = (orders.withColumn("r", fn().over(w)).filter(col("r") == 1).count())
    print(f"{fn_name:>11} == 1 -> {n} rows (one per category is {orders.select('category').distinct().count()})")

## Top-N per group — the pattern you'll write forever

"Latest order per customer." Note the tiebreaker: order_date alone
is NOT unique (several customers order twice on different dates but
the data has same-date pairs too) — order_id finishes the job.

In [ ]:
w_latest = Window.partitionBy("customer_id").orderBy(
    F.desc("order_date"), F.desc("order_id"))

latest = (orders
          .withColumn("rn", F.row_number().over(w_latest))
          .filter(col("rn") == 1)
          .drop("rn"))
print("customers with orders:", latest.count())
latest.select("customer_id", "order_id", "order_date", "product").orderBy("customer_id").show(6)

In [ ]:
# rank()==1 answers a DIFFERENT question: all orders tied for latest date.
# For c012 the duplicated 1021 rows share the max date -> both come back.
w_date_only = Window.partitionBy("customer_id").orderBy(F.desc("order_date"))
tied_latest = (orders
               .withColumn("r", F.rank().over(w_date_only))
               .filter(col("r") == 1))
print("row_number version:", latest.count(), "| rank version:", tied_latest.count())
tied_latest.filter(col("customer_id") == "c012").select("customer_id", "order_id", "order_date").show()

In [ ]:
# top-3 price points per category — dense_rank, because "3rd highest VALUE"
w_price = Window.partitionBy("category").orderBy(F.desc("unit_price"))
(orders
 .withColumn("price_tier", F.dense_rank().over(w_price))
 .filter(col("price_tier") <= 3)
 .select("category", "product", "unit_price", "price_tier")
 .orderBy("category", "price_tier")
 .show(12))

## The nondeterminism demo

Same data, same code, shuffled input order — row_number without a
full tiebreaker can flip which twin wins. (On tiny local data both
runs often agree by luck; the point is that NOTHING promises it.)

In [ ]:
dup_rows = orders.filter(col("order_id") == 1021)   # the identical twins
w_bad = Window.partitionBy("customer_id").orderBy(F.desc("order_date"))       # ties!
w_good = Window.partitionBy("customer_id").orderBy(F.desc("order_date"),
                                                   F.desc("order_id"))        # unique... except for true dups
dup_rows.withColumn("rn_bad", F.row_number().over(w_bad)).select(
    "order_id", "order_date", "rn_bad").show()
# true duplicate ROWS can't be ordered by any column — deduplicate them
# first (3.3), or accept arbitrary assignment knowingly. 8.4 returns here.

## The plan: WindowGroupLimit (Spark 3.5+)

The rn<=k filter is pushed into the sort — executors keep only k
rows per group instead of ranking everything, same code.

In [ ]:
latest.explain()
# look for WindowGroupLimit above the Sort/Exchange

## Exercises

1. "Second-highest revenue order per country" — handle both tie
   interpretations: exactly-one-row (row_number) and
   all-rows-at-that-value (dense_rank). Nulls in country: keep or
   drop? Decide and defend.
2. ntile(4) over all 41 orders by revenue: predict the four bucket
   sizes before running. Then find a pair of equal revenues split
   across buckets and fix the banding with a value-based `when`
   ladder instead.
3. Using percent_rank, label customers 'top5', 'top20', 'rest' by
   lifetime spend (build customer totals first — groupBy or window,
   your call, but count the shuffles either way).
4. Write the classic SQL interview query in spark.sql(): top 2
   products per category by total revenue, ROW_NUMBER() OVER in a
   subquery — and confirm the DataFrame and SQL plans match.

In [ ]:
# your exercise code here